In [1]:
import pandas as pd
import torch
import numpy as np


states = torch.load("states.pt")
target = pd.read_csv("target.csv")

In [2]:
DEVICE = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"

In [3]:
row = states.shape[0]

print(states.shape)
print(target)

torch.Size([707542, 30, 8, 8])
        value  policy
0          -1     307
1           1    4488
2          -1     657
3           1    4395
4          -1     195
...       ...     ...
707537      1    2538
707538     -1    4488
707539      1    2847
707540     -1    2604
707541      1    3480

[707542 rows x 2 columns]


In [4]:
import sys
sys.path.append('..')

In [5]:
policy = torch.tensor(target.policy.values).float()
value = torch.tensor(target.value.values).float()


In [10]:
# Train test split
TRAIN_SIZE = int(0.9 * len(states)) 

train_states, test_states = states[:TRAIN_SIZE], states[TRAIN_SIZE:]
train_policy, test_policy = policy[:TRAIN_SIZE], policy[TRAIN_SIZE:]
train_value,  test_value  = value[:TRAIN_SIZE],  value[TRAIN_SIZE:]


In [11]:

from core import factory

network = factory.build_network("chess")

In [12]:
from torch.optim import Adam
from torch.nn import CrossEntropyLoss, MSELoss

optimizer = Adam(network.parameters(), lr=1e-3, fused=True)
value_loss_fn = MSELoss()
policy_loss_fn = CrossEntropyLoss()

In [17]:
from core.network import PolicyValueNetwork
from torch import optim
import time
from core.network import PolicyValueNetwork
from torch import optim
import time
import torch
import numpy as np
from torch.optim import lr_scheduler


def evaluate(network, test_states, test_policy, test_value,
             policy_loss_fn, value_loss_fn, batch_size=256):
    network.eval()
    total_policy_loss, total_value_loss, n_batches = 0.0, 0.0, 0

    with torch.no_grad():
        for i in range(0, len(test_states), batch_size):
            batch_states = test_states[i:i+batch_size]
            batch_policy = test_policy[i:i+batch_size]
            batch_value  = test_value[i:i+batch_size]

            policy_head, value_head = network(batch_states)
            total_policy_loss += policy_loss_fn(policy_head, batch_policy).item()
            total_value_loss  += value_loss_fn(value_head, batch_value).item()
            n_batches += 1

    network.train()
    return total_policy_loss / n_batches, total_value_loss / n_batches


def train(network: PolicyValueNetwork,
          optimizer: optim.Optimizer,
          train_states: torch.Tensor,
          train_policy: torch.Tensor,
          train_value: torch.Tensor,
          policy_loss_fn,
          value_loss_fn,
          test_states: torch.Tensor | None = None,
          test_policy: torch.Tensor | None = None,
          test_value: torch.Tensor | None = None,
          batch_size: int = 256,
          num_iter: int | None = None,
          duration_hour: float | None = None,
          eval_every: int = 500,
          seed: int = 42):

    if num_iter is None and duration_hour is None:
        raise ValueError("Must specify at least one of num_iter or duration_hour")

    start = time.time()
    rng = np.random.default_rng(seed=seed)
    step = 0

    scheduler = lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=num_iter if num_iter is not None else 9999)

    train_policy = train_policy.to(device=DEVICE)
    train_value = train_value.unsqueeze(-1).to(device=DEVICE)

    has_test = test_states is not None and test_policy is not None and test_value is not None
    if has_test:
        test_states = test_states.to(device=DEVICE)
        test_policy = test_policy.to(device=DEVICE)
        test_value  = test_value.unsqueeze(-1).to(device=DEVICE)

    while True:
        if num_iter is not None and step >= num_iter:
            break
        if duration_hour is not None and time.time() - start >= duration_hour * 3600:
            break

        batch_idx = rng.choice(len(train_states), batch_size, replace=False)
        batch_train_states = train_states[batch_idx]
        batch_train_policy = train_policy[batch_idx]
        batch_train_value  = train_value[batch_idx]

        optimizer.zero_grad()
        policy_head, value_head = network(batch_train_states)

        policy_loss = policy_loss_fn(policy_head, batch_train_policy)
        value_loss  = value_loss_fn(value_head, batch_train_value)
        loss = policy_loss + value_loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        if step % 10 == 0:
            elapsed = time.time() - start
            print(f"[{step}] loss={loss.item():.4f} | policy={policy_loss.item():.4f} | value={value_loss.item():.4f} | {elapsed:.0f}s")

        if has_test and step % eval_every == 0 and step > 0:
            val_policy_loss, val_value_loss = evaluate(
                network, test_states, test_policy, test_value, policy_loss_fn, value_loss_fn
            )
            print(f"    [eval @ {step}] val_policy={val_policy_loss:.4f} | val_value={val_value_loss:.4f}")

        step += 1

In [ ]:
n_train = int(0.8 * len(states))
train_states, test_states = states[:n_train], states[n_train:]
train_policy, test_policy = policy[:n_train], policy[n_train:]
train_value,  test_value  = value[:n_train],  value[n_train:]

train(
    duration_hour=1,
    network=network,
    optimizer=optimizer,
    train_states=train_states,
    train_policy=train_policy,
    train_value=train_value,
    test_states=test_states,
    test_policy=test_policy,
    test_value=test_value,
    policy_loss_fn=policy_loss_fn,
    value_loss_fn=value_loss_fn,
    batch_size=32,
    seed=42
)

[0] loss=5.4886 | policy=5.1182 | value=0.3704 | 1s
[10] loss=6.3675 | policy=5.9331 | value=0.4344 | 5s
[20] loss=6.3342 | policy=5.7790 | value=0.5552 | 9s
[30] loss=5.8231 | policy=5.2797 | value=0.5434 | 13s
[40] loss=5.9561 | policy=5.3709 | value=0.5853 | 17s
[50] loss=7.0923 | policy=6.5330 | value=0.5593 | 22s
[60] loss=6.8038 | policy=6.2954 | value=0.5084 | 26s
[70] loss=7.3629 | policy=6.8415 | value=0.5213 | 30s
[80] loss=7.0683 | policy=6.5594 | value=0.5089 | 35s
[90] loss=6.9091 | policy=6.5420 | value=0.3671 | 39s
[100] loss=6.6536 | policy=6.1468 | value=0.5067 | 43s
[110] loss=6.3669 | policy=5.8240 | value=0.5429 | 47s
[120] loss=6.7967 | policy=6.3350 | value=0.4617 | 51s
[130] loss=7.0250 | policy=6.4828 | value=0.5423 | 55s
[140] loss=6.8171 | policy=6.2576 | value=0.5595 | 59s
[150] loss=6.3834 | policy=6.1060 | value=0.2774 | 63s
[160] loss=6.2964 | policy=5.9358 | value=0.3606 | 68s
[170] loss=6.6994 | policy=6.1891 | value=0.5102 | 72s
[180] loss=6.6812 | poli